# 02 — Regressão logística: versão mínima

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flavioluizseixas/aprendizado-de-maquina-para-saude/blob/main/notebooks/02_aprendizado_supervisionado_vers%C3%A3o_minima.ipynb)

Execute no Colab ou em Jupyter com `numpy`, `pandas`, `scikit-learn` e `statsmodels` instalados. Se necessário, execute `%pip install statsmodels` em uma célula antes da seção 5.3. O carregamento requer internet.

## Fonte e licença

[CDC Diabetes Health Indicators — UCI 891](https://archive.ics.uci.edu/dataset/891/cdc+diabetes+health+indicators). Consulte a fonte para os códigos das variáveis e os termos de uso.

**Alvo:** 0 = sem diabetes; 1 = pré-diabetes/diabetes. Para simplificar, os códigos ordinais de `Age`, `GenHlth`, `Education` e `Income` permanecem numéricos.

Material com finalidade exclusivamente educacional.

## 1. Dados e split

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score

dados = pd.read_csv("https://archive.ics.uci.edu/static/public/891/data.csv", index_col="ID")
X, y = dados.drop(columns="Diabetes_binary"), dados["Diabetes_binary"].astype(int)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

## 2. Padronização e treinamento

O pipeline aprende a média, o desvio-padrão e os coeficientes somente no treino. `predict` usa limiar 0,5.

In [ ]:
modelo = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
modelo.fit(X_train, y_train)
y_pred = modelo.predict(X_test)
y_prob = modelo.predict_proba(X_test)[:, 1]

## 3. Desempenho no teste

No relatório, `recall` da classe 1 é a sensibilidade; da classe 0, a especificidade. Observe também precisão, F1 e acurácia.

In [ ]:
print(classification_report(y_test, y_pred, digits=3))
print("ROC-AUC:", round(roc_auc_score(y_test, y_prob), 3))

## 4. Coeficientes da regressão logística

Coeficiente positivo aumenta a log-odds estimada da classe 1; negativo a reduz, mantendo os demais atributos constantes.

Os coeficientes usam a **escala padronizada**: nas variáveis contínuas, uma unidade corresponde a um desvio-padrão do treino. Nas binárias, a mudança de 0 para 1 não equivale a um desvio-padrão. São coeficientes de um modelo regularizado; não representam efeitos causais.

In [ ]:
pd.Series(modelo[-1].coef_[0], index=X.columns, name="Coeficiente").sort_values().round(3)

## 5. Odds e odds ratio a partir dos coeficientes estimados

### 5.1. Da log-odds à probabilidade

Para a probabilidade estimada $p$ da classe 1, **odds** é a razão $p/(1-p)$. Por exemplo, $p=0{,}20$ corresponde a odds de $0{,}25$: uma ocorrência da classe 1 para cada quatro da classe 0, em termos probabilísticos. Odds não é uma porcentagem.

O modelo ajustado calcula, com $z_j=(x_j-\mu_j)/s_j$:

$$\eta = \log\left(\frac{p}{1-p}\right) = \beta_0 + \sum_j \beta_j z_j, \qquad \mathrm{odds}=e^{\eta}, \qquad p=\frac{\mathrm{odds}}{1+\mathrm{odds}}.$$

Aqui, $\mu_j$ e $s_j$ são a média e a escala aprendidas pelo `StandardScaler` no treino. Para usar os valores originais $x_j$, convertemos **também o intercepto**:

$$b_j=\frac{\beta_j}{s_j}, \qquad b_0=\beta_0-\sum_j b_j\mu_j, \qquad \eta=b_0+\sum_j b_jx_j.$$

A célula abaixo reaproveita o modelo treinado e calcula odds e probabilidades para os cinco primeiros registros do teste. Compare as probabilidades reconstruídas com `predict_proba`. Os cálculos usam os coeficientes completos; arredondamos somente a exibição.

In [ ]:
padronizador, regressao = modelo[0], modelo[-1]
coef_pad = pd.Series(regressao.coef_[0], index=X.columns)
escala = pd.Series(padronizador.scale_, index=X.columns)
coef_original = coef_pad / escala
intercepto_original = regressao.intercept_[0] - (coef_original * padronizador.mean_).sum()

perfis = X_test.head(5)
log_odds = intercepto_original + perfis @ coef_original
odds = np.exp(log_odds)
prob_calculada = odds / (1 + odds)

print(f"Intercepto na escala original: {intercepto_original:.4f}")
display(pd.DataFrame({
    "Log-odds": log_odds,
    "Odds da classe 1": odds,
    "Probabilidade calculada": prob_calculada,
    "Probabilidade do pipeline": modelo.predict_proba(perfis)[:, 1],
}).round(4))

### 5.2. Odds ratio: qual mudança está sendo comparada?

Mantendo os demais atributos constantes, para um aumento $\Delta x_j$ na variável original:

$$\mathrm{OR}=\frac{\mathrm{odds}(x_j+\Delta x_j)}{\mathrm{odds}(x_j)}=\exp\left(\frac{\beta_j}{s_j}\Delta x_j\right)=\exp(b_j\Delta x_j).$$

Assim, `exp(coef_pad)` descreve um aumento de **uma unidade padronizada** (um desvio-padrão do treino em atributos não constantes); `exp(coef_original)` descreve **uma unidade original**. Em `HighBP`, a mudança de 0 para 1 compara presença com ausência de pressão alta. Em `BMI`, +1 significa +1 kg/m² de IMC.

- **OR > 1:** odds maiores; **OR < 1:** odds menores; **OR = 1:** odds iguais no contraste do modelo.
- A variação percentual das odds é $100(\mathrm{OR}-1)$. Por exemplo, OR = 1,50 indica odds 50% maiores; OR = 0,70 indica odds 30% menores. Isso não é a variação percentual da probabilidade nem um risco relativo.
- Nas binárias, interprete a OR de **0 para 1**, e não a de um desvio-padrão. Em `Age`, `GenHlth`, `Education` e `Income`, +1 significa um passo no código ordinal, não um ano ou uma unidade monetária. A codificação numérica assume o mesmo incremento na log-odds em cada passo.

A primeira tabela mostra as duas escalas. A segunda destaca três contrastes para interpretar em palavras. Cada contraste altera somente a variável indicada.

In [ ]:
tabela_or = pd.DataFrame({
    "Coef. padronizado": coef_pad,
    "Escala do treino": escala,
    "Coef. original": coef_original,
    "OR (+1 padronizada)": np.exp(coef_pad),
    "OR (+1 original)": np.exp(coef_original),
}).rename_axis("Atributo")
display(tabela_or.sort_values("OR (+1 original)", ascending=False).round(4))

contrastes = pd.DataFrame({
    "Atributo": ["BMI", "BMI", "HighBP"],
    "Comparação": ["IMC: +1 kg/m²", "IMC: +5 kg/m²", "Pressão alta: 1 versus 0"],
    "Incremento": [1, 5, 1],
})
contrastes["OR"] = np.exp(
    contrastes["Atributo"].map(coef_original) * contrastes["Incremento"]
)
contrastes["Variação das odds (%)"] = 100 * (contrastes["OR"] - 1)
display(contrastes.round(4))

### Interpretação e limites

Complete para cada contraste: **«Mantendo os demais atributos constantes, [mudança] está associada a odds estimadas da classe 1 multiplicadas por [OR], ou [percentual]% maiores/menores, segundo este modelo.»**

Para um exercício separado, suponha $p_0=0{,}20$ e uma OR hipotética de 2. As odds passam de 0,25 para 0,50 e $p_1=0{,}50/(1+0{,}50)\approx0{,}333$. Dobrar as odds não dobra a probabilidade. Em geral, $p_1=\mathrm{OR}\,p_0/(1-p_0+\mathrm{OR}\,p_0)$.

Estas são OR condicionais do **mesmo modelo preditivo regularizado**, sem novo ajuste. Converter a escala não remove a regularização. A seção seguinte calcula IC95% e valores-p em um ajuste sem penalização, com suas próprias OR. Os dados são observacionais e incluem autorrelato; as associações não demonstram causalidade nem validam uso clínico.

**Para responder:** por que é necessário dividir o coeficiente padronizado pela escala antes de interpretar `HighBP` de 0 para 1? Como a OR para +5 unidades de IMC se relaciona com a OR para +1? Por que uma OR, sozinha, não informa a probabilidade de uma pessoa?

**Referências:** [Penn State — regressão logística e odds ratio](https://online.stat.psu.edu/stat462/node/207/); documentação de [StandardScaler](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html) e [LogisticRegression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html).

### 5.3. Tabela final: odds ratio, IC95% e valor-p

Para acrescentar significância estatística, ajustamos uma **regressão logística sem penalização** (`statsmodels.GLM`, família binomial e ligação logit), com intercepto, os mesmos 21 atributos e **somente os dados de treino**. Mantemos as unidades originais e a mesma codificação numérica das categorias ordinais.

O ajuste é identificado como `modelo_inferencial`. Seus coeficientes, OR, IC95% e valores-p pertencem todos a esse ajuste; podem diferir dos valores do pipeline regularizado. O pipeline continua sendo o modelo usado nas métricas e nas probabilidades anteriores.

Usamos erros-padrão robustos **HC0**, teste de Wald bilateral com aproximação normal e nível de significância nominal de 5%. Para cada atributo, testamos $H_0:b_j=0$, equivalente a $\mathrm{OR}=1$ para +1 unidade original:

$$z_j=\frac{\hat b_j}{\mathrm{EP}_{HC0}(\hat b_j)}, \qquad \mathrm{IC95\%}(\mathrm{OR}_j)=\exp\left[\hat b_j\pm1{,}96\,\mathrm{EP}_{HC0}(\hat b_j)\right].$$

In [ ]:
import statsmodels.api as sm

X_inferencia = sm.add_constant(X_train.astype(float), has_constant="add")
modelo_inferencial = sm.GLM(
    y_train, X_inferencia, family=sm.families.Binomial(), missing="raise"
).fit(cov_type="HC0", use_t=False)
if not modelo_inferencial.converged:
    raise RuntimeError("O modelo inferencial não convergiu; revise o ajuste antes de interpretar.")

coef_inferencial = modelo_inferencial.params.loc[X.columns]
ic_coef = modelo_inferencial.conf_int(alpha=0.05).loc[X.columns]
valores_p = modelo_inferencial.pvalues.loc[X.columns]
tabela_inferencia = pd.DataFrame({
    "Coef. original": coef_inferencial,
    "Erro-padrão HC0": modelo_inferencial.bse.loc[X.columns],
    "OR (+1 original)": np.exp(coef_inferencial),
    "IC95% OR inferior": np.exp(ic_coef[0]),
    "IC95% OR superior": np.exp(ic_coef[1]),
    "Valor-p (Wald)": valores_p,
    "Significativo a 5%": np.where(valores_p < 0.05, "Sim", "Não"),
}).rename_axis("Atributo")

print(f"Ajuste sem penalização: {len(y_train):,} registros de treino; {y_train.sum():,} da classe 1.")
display(
    tabela_inferencia.sort_values("OR (+1 original)", ascending=False).style.format(
        {"Valor-p (Wald)": lambda p: "< 1e-300" if p < 1e-300 else f"{p:.2e}"},
        precision=4,
    ).set_caption("Regressão logística sem penalização — IC95% e Wald robustos HC0")
)

**Como interpretar a significância:**

- **p < 0,05:** evidência contra $H_0$ no teste adotado. Leia a direção e a magnitude na OR e a incerteza no IC95%; significância estatística não mede importância clínica.
- **p ≥ 0,05:** evidência insuficiente para rejeitar $H_0$ a 5%; não prova que a associação seja inexistente. Um IC95% que inclui 1 é compatível com esse resultado, salvo arredondamento na exibição.
- O valor-p é calculado sob $H_0$ e as hipóteses do teste; não é a probabilidade de $H_0$ ser verdadeira. O indicador de significância usa o valor-p completo, antes do arredondamento.

**Limites:** os testes são exploratórios, sem correção para múltiplas comparações. A inferência pressupõe observações independentes e não incorpora pesos, estratos ou conglomerados do BRFSS; HC0 não substitui uma análise do desenho amostral. A codificação ordinal, a linearidade na log-odds e a ausência de interações continuam sendo hipóteses do modelo.

**Referências:** documentação de [GLM e ajuste da covariância](https://www.statsmodels.org/stable/generated/statsmodels.genmod.generalized_linear_model.GLM.fit.html), [valores-p](https://www.statsmodels.org/stable/generated/statsmodels.genmod.generalized_linear_model.GLMResults.pvalues.html) e [intervalos de confiança](https://www.statsmodels.org/stable/generated/statsmodels.genmod.generalized_linear_model.GLMResults.conf_int.html).